In [ ]:
!pip install neuralforecast

# **AutoFormer**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from neuralforecast import NeuralForecast
from neuralforecast.models import Autoformer
from neuralforecast.utils import AirPassengersPanel, AirPassengersStatic, augment_calendar_df

from neuralforecast.losses.pytorch import MAE
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
AirPassengersPanel, calendar_cols = augment_calendar_df(df=AirPassengersPanel, freq='M')

In [ ]:
AirPassengersPanel.head(3)

In [ ]:
Y_train_df = AirPassengersPanel[AirPassengersPanel.ds<AirPassengersPanel['ds'].values[-12]] # 132 train
Y_test_df = AirPassengersPanel[AirPassengersPanel.ds>=AirPassengersPanel['ds'].values[-12]].reset_index(drop=True) # 12 test

In [ ]:
Y_train_df.head(3)

In [ ]:
Y_test_df_copy = Y_test_df.copy(deep = True)
Y_test_df_copy['y'] = 0
Y_test_df_copy.head(3)

In [ ]:
model = Autoformer(h=12,                            # Forecast horizon (how many future steps)
                 input_size=24,                     # How much past data model sees, Here: last 24 months (2 years)
                 hidden_size = 16,                  # Model “brain size” (embedding dimension), Small dataset → 16–32, Medium → 64–128, Large → 256+
                 conv_hidden_size = 32,             # Size of convolution layer (used in decomposition). Rule: Usually 2× hidden_size
                 n_head=2,                          # Number of heads in Auto-Correlation. More heads → capture different patterns. How to choose: Must divide hidden_size
                 loss=MAE(),                        # Loss Function. Other Example: MSE(), MAPE()
                 futr_exog_list=calendar_cols,      # Future known variables (VERY powerful). How to choose: Include only variables that: are known in futur & influence target
                 scaler_type='robust',              # Scales data using median + IQR. Less sensitive to outliers. Other example: 'standard', 'robust', 'minmax'
                 learning_rate=1e-3,                # Step size for learning
                 max_steps=300,                     # Total training steps. Small data → 200–500, Large data → 1000+
                 val_check_steps=50,                # Check validation every 50 steps. Rule: ~5–10 checks per training. 300 / 50 = 6 checks ✅ good
                 early_stop_patience_steps=2)       # Stop if no improvement for 2 validation checks. Small data → 3–5, Large data → 5–10

nf = NeuralForecast(
    models=[model],
    freq='ME'
)

nf.fit(df=Y_train_df, static_df=AirPassengersStatic, val_size=12)
forecasts_auto = nf.predict(futr_df=Y_test_df_copy)

In [ ]:
Y_hat_df = forecasts_auto.reset_index(drop=False).drop(columns=['unique_id','ds'])
plot_df = pd.concat([Y_test_df, Y_hat_df], axis=1)
plot_df = pd.concat([Y_train_df, plot_df])
plot_df.tail(3)

In [ ]:
if model.loss.is_distribution_output:
    plot_df = plot_df[plot_df.unique_id=='Airline1'].drop('unique_id', axis=1)
    plt.plot(plot_df['ds'], plot_df['y'], c='black', label='True')
    plt.plot(plot_df['ds'], plot_df['Autoformer-median'], c='blue', label='median')
    plt.fill_between(x=plot_df['ds'][-12:],
                    y1=plot_df['Autoformer-lo-90'][-12:].values,
                    y2=plot_df['Autoformer-hi-90'][-12:].values,
                    alpha=0.4, label='level 90')
    plt.grid()
    plt.legend()
    plt.plot()
else:
    plot_df = plot_df[plot_df.unique_id=='Airline1'].drop('unique_id', axis=1)
    plt.plot(plot_df['ds'], plot_df['y'], c='black', label='True')
    plt.plot(plot_df['ds'], plot_df['Autoformer'], c='blue', label='Forecast')
    plt.legend()
    plt.grid()

In [ ]:
print("MAPE AutoFormer: ", mean_absolute_percentage_error(Y_test_df['y'], forecasts_auto['Autoformer']))